In [2]:
!pip install pyspark

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# start the spark engine
spark = SparkSession.builder\
        .appName("SalesDataLAke")\
        .getOrCreate()


In [8]:
#look at the Messy Data

# read raw data from Bronze
bronze_path = "data_lake/bronze/sales.csv"
df_bronze = spark.read.csv(bronze_path,  header=True, inferSchema=True)
#check structure and row count
df_bronze.printSchema()
df_bronze.show(5)
print("Total Bronze Rows:", df_bronze.count())

root
 |-- transaction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- transaction_date: date (nullable = true)

+--------------+-----------+------------+--------+----------------+
|transaction_id|customer_id|product_name|  amount|transaction_date|
+--------------+-----------+------------+--------+----------------+
|             1|        101|      Laptop| 1200.00|      2023-10-01|
|             2|        102|  Smartphone| 800.50 |      2023-10-02|
|             3|        103|  Headphones|   99.99|      2023-10-03|
|             4|       NULL|       Mouse|   25.00|      2023-10-04|
|             5|        105|    Keyboard|    null|      2023-10-05|
+--------------+-----------+------------+--------+----------------+

Total Bronze Rows: 5


In [7]:
import os

# Create the bronze directory structure
os.makedirs("data_lake/bronze", exist_ok=True)

# Generate a sample messy CSV file
sample_data = """transaction_id,customer_id,product_name,amount,transaction_date\n1,101,Laptop,1200.00,2023-10-01\n2,102,Smartphone, 800.50 ,2023-10-02\n3,103,Headphones,99.99,2023-10-03\n4,,Mouse,25.00,2023-10-04\n5,105,Keyboard,null,2023-10-05\n"""

with open("data_lake/bronze/sales.csv", "w") as f:
    f.write(sample_data.strip())

print("Directory and mock sales.csv file created successfully!")

Directory and mock sales.csv file created successfully!


The dataset header has a tiny typo (discount_percen) adn the data contains negative numbers like -2 quantity or -500 pricw discounts over 100%,inconsistent capitalization HYDERABAD,delhi, Sports and weired dates 31/02/2026, not-a-date,2026-13-05.

In [12]:
# 1. count exact duplicates
total_rows = df_bronze.count()
unique_rows = df_bronze.dropDuplicates().count()
print("Exact Duplicates:", total_rows - unique_rows)

# 2. Count missing (null) values in every column
null_counts = df_bronze.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_bronze.columns
])
print("Missing Values per column:")
null_counts.show()

# 3. Check invalid numbers (e.g., negative or non-numeric values in the 'amount' column)
# In PySpark, try_cast is a method called directly on the Column object: col.try_cast(...)
safe_double_amount = F.col("amount").try_cast("double")

invalid_nums = df_bronze.filter(
    (safe_double_amount.isNull() & F.col("amount").isNotNull() & (F.trim(F.col("amount")) != "null")) |
    (safe_double_amount <= 0)
).count()

print("Rows with invalid or negative amounts:", invalid_nums)

Exact Duplicates: 0
Missing Values per column:
+--------------+-----------+------------+------+----------------+
|transaction_id|customer_id|product_name|amount|transaction_date|
+--------------+-----------+------------+------+----------------+
|             0|          1|           0|     0|               0|
+--------------+-----------+------------+------+----------------+

Rows with invalid or negative amounts: 0


In [14]:
df_silver = df_bronze

# 1. Remove exact duplicate rows
df_silver = df_silver.dropDuplicates()

# 2. Handle missing values (fill missing text with 'Unknown')
df_silver = df_silver.fillna({
    "product_name": "Unknown"
})

# 3. Trim extra spaces and fix capitalization (Title Case) for product name
df_silver = df_silver.withColumn("product_name", F.initcap(F.trim(F.col("product_name"))))

# 4. Fix and sanitize amounts
# Safely convert 'amount' string to double, keeping only positive numbers
safe_amount = F.col("amount").try_cast("double")
df_silver = df_silver.withColumn("amount", safe_amount)
df_silver = df_silver.filter((F.col("amount").isNotNull()) & (F.col("amount") > 0))

# 5. Drop rows with invalid or missing transaction dates
df_silver = df_silver.filter(F.col("transaction_date").isNotNull())

# Display cleaned Silver data
print("Cleaned Silver Data:")
df_silver.show()

Cleaned Silver Data:
+--------------+-----------+------------+------+----------------+
|transaction_id|customer_id|product_name|amount|transaction_date|
+--------------+-----------+------------+------+----------------+
|             1|        101|      Laptop|1200.0|      2023-10-01|
|             3|        103|  Headphones| 99.99|      2023-10-03|
|             2|        102|  Smartphone| 800.5|      2023-10-02|
|             4|       NULL|       Mouse|  25.0|      2023-10-04|
+--------------+-----------+------------+------+----------------+



In [15]:
# Save to Silver folder
silver_folder = "data_lake/silver/preprocessed_sales"
df_silver.write.mode("overwrite").option("header", True).csv(silver_folder)

# Read back to verify
df_verified = spark.read.csv(silver_folder, header=True, inferSchema=True)

print("Row count BEFORE preprocessing:", df_bronze.count())
print("Row count AFTER preprocessing:", df_verified.count())
df_verified.show(10)

Row count BEFORE preprocessing: 5
Row count AFTER preprocessing: 4
+--------------+-----------+------------+------+----------------+
|transaction_id|customer_id|product_name|amount|transaction_date|
+--------------+-----------+------------+------+----------------+
|             1|        101|      Laptop|1200.0|      2023-10-01|
|             3|        103|  Headphones| 99.99|      2023-10-03|
|             2|        102|  Smartphone| 800.5|      2023-10-02|
|             4|       NULL|       Mouse|  25.0|      2023-10-04|
+--------------+-----------+------------+------+----------------+



In [16]:
df_verified.toPandas().to_csv("sales_silver.csv", index=False)